# 06 (Databricks). Modele ML predictive - Pret energie Spania

**Versiune adaptata pentru Databricks** a notebook-ului `06_ml_pret_spania.ipynb`.

**Sesiunea 2 din Etapa II: Dezvoltarea Modelelor Predictive (Machine Learning)**

**Scop:** anticiparea pretului orar al energiei electrice (`price actual`) in piata spaniola, 2015-2018, rulat **full** in cloud pentru rezultate finale (LSTM si Optuna cu parametri completi).

**Diferente fata de versiunea locala (06):**
1. **MLflow tracking** - fiecare model e logat automat (parametri, metrici, model serializat). Vezi tab-ul "Experiments".
2. **Download cod + date** din Workspace Files via REST API in `/tmp/` (Workspace Files nu sunt accesibile direct prin `open()`).
3. **Detectie GPU** - LSTM foloseste GPU automat daca exista.
4. **%pip install** - librariile lipsa pe runtime general (xgboost, optuna, shap, tensorflow).

**Inovatiile metodologice ale Sesiunii 2** (fata de USA):
- **Optuna** (Bayesian optimization, TPE) in loc de GridSearchCV - mai eficient pe spatii mari de hyperparametri.
- **SHAP values** pentru explicabilitate - cu 80 features, contributia fiecarei variabile e critica.

**Algoritmi comparati:** LinearRegression (baseline), **Ridge** (L2), **Lasso** (L1), RandomForest, XGBoost (default + Optuna-tuned), LSTM. **Prophet exclus** (a esuat la USA pe acest tip de date).

**Inainte de a rula:** vezi `docs/DATABRICKS_SETUP.md` (cluster, Repo, upload date in `data/processed/`).

## Setup - instalare dependinte

In [ ]:
# Daca cluster-ul e cu ML Runtime, multe sunt deja instalate ("Requirement already satisfied").
# Pe runtime general (16.4 LTS fara ML) le instaleaza pe toate.
# Spania NU foloseste Prophet (exclus), dar foloseste optuna + shap.
#
# IMPORTANT: versiuni FIXATE pe numpy<2 si protobuf<5 ca sa NU stricam runtime-ul Databricks.
# Fara pin, pip aduce numpy 2.x + protobuf 6.x -> kernel-ul nu mai porneste la restartPython().
# tensorflow==2.16.1 e ultima versiune care cere numpy<2 (compatibila cu Databricks 16.4 LTS).
%pip install --quiet "tensorflow==2.16.1" "numpy<2" "protobuf<5" mlflow xgboost optuna shap scikit-learn pyyaml

In [ ]:
dbutils.library.restartPython()

## Imports si setup

In [ ]:
# === Setup imports + descarcare cod din Workspace Files ===
import sys, os, time, json, base64
import urllib.request, urllib.parse
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import mlflow.sklearn
import mlflow.tensorflow

# Workspace Files in /Workspace/Users/ NU sunt accesibile prin Python open().
# Solutia: download via Databricks REST API si salvare in /tmp/ (file system normal).

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
API_TOKEN = ctx.apiToken().get()
API_URL = ctx.apiUrl().get()
NOTEBOOK_PATH = ctx.notebookPath().get()

# Repo path = parent al folder-ului notebooks (notebook e in notebooks/)
WORKSPACE_REPO_PATH = '/'.join(NOTEBOOK_PATH.split('/')[:-2])
print(f'Workspace repo path: {WORKSPACE_REPO_PATH}')

LOCAL_BASE = '/tmp/disertatie_ai_platform'

def download_workspace_file(workspace_path, local_path):
    encoded = urllib.parse.quote(workspace_path)
    url = f'{API_URL}/api/2.0/workspace/export?path={encoded}&format=SOURCE'
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {API_TOKEN}'})
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read())
    content = base64.b64decode(data['content'])
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    with open(local_path, 'wb') as f:
        f.write(content)

files_to_download = [
    'src/__init__.py',
    'src/utils/__init__.py',
    'src/utils/config_loader.py',
    'src/utils/plotting.py',
    'src/data_processing/__init__.py',
    'src/data_processing/preprocessing.py',
    'src/data_processing/loader.py',
    'src/ml_models/__init__.py',
    'src/ml_models/predictors.py',
    'config.yaml',
]

print(f'Descarcare {len(files_to_download)} fisiere din Workspace in {LOCAL_BASE}...')
for f_rel in files_to_download:
    ws_full = f'{WORKSPACE_REPO_PATH}/{f_rel}'
    local_full = f'{LOCAL_BASE}/{f_rel}'
    try:
        download_workspace_file(ws_full, local_full)
        print(f'  OK: {f_rel}')
    except Exception as e:
        print(f'  FAIL: {f_rel} - {e}')

if LOCAL_BASE not in sys.path:
    sys.path.insert(0, LOCAL_BASE)

PROJECT_ROOT = Path(LOCAL_BASE)
print(f'\nPROJECT_ROOT (local cache): {PROJECT_ROOT}')

# === Import-uri (codul e acum in /tmp/) ===
from sklearn.linear_model import Ridge, Lasso
from src.data_processing.preprocessing import chronological_split, scale_features
from src.ml_models.predictors import (
    evaluate, train_linear, train_random_forest, train_xgboost,
    train_lstm, predict_lstm,
    time_series_cv, tune_with_optuna, compute_shap_values,
    save_model, get_feature_importance, ModelResult,
)
from src.utils.config_loader import load_config
from src.utils.plotting import setup_style, PALETA

setup_style()
warnings.filterwarnings('ignore')

cfg = load_config(PROJECT_ROOT / 'config.yaml')
P_CFG = cfg['preprocessing']
print('Imports OK. Setup complet.')

## Detectie GPU

Daca cluster-ul are GPU (Standard_NC* in Azure, g4dn.* in AWS), TensorFlow il foloseste automat, accelerand LSTM cu ~10x fata de CPU.

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs disponibile: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

if not gpus:
    print("\n[INFO] Niciun GPU detectat - LSTM va rula pe CPU.")
    print("       Pentru viteza, recomand cluster cu GPU (Single Node, T4 sau similar).")
else:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print("\n[OK] GPU configurat pentru LSTM.")

## MLflow tracking

**Ce este MLflow?** Platforma open-source integrata in Databricks pentru tracking experimente ML.

**Ce face automat:** salveaza fiecare rulare cu hyperparametrii folositi, metricile obtinute (RMSE, MAE, R^2, MAPE), si serializeaza modelul antrenat. Le poti compara side-by-side in tab-ul "Experiments" (iconita din panoul lateral stang).

Pentru lucrare, MLflow demonstreaza o **practica reala de productie** (experiment tracking), exact ce cauta angajatorii pentru pozitii de Data Scientist / ML Engineer.

In [ ]:
EXPERIMENT_NAME = "/Users/" + ctx.userName().get() + "/disertatie_pret_spania"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")

## 1. Incarcare date - detectare automata path

Notebook-ul incearca mai multe strategii pentru a gasi `pret_spania_features.parquet`: copiere din Workspace Files (repo), apoi fallback in DBFS legacy / Unity Catalog Volumes. Vezi `docs/DATABRICKS_SETUP.md` daca niciuna nu reuseste.

In [ ]:
# === Load data parquet ===
parquet_filename = 'pret_spania_features.parquet'
parquet_local = LOCAL_BASE + '/data/processed/' + parquet_filename

if not os.path.exists(parquet_local):
    os.makedirs(os.path.dirname(parquet_local), exist_ok=True)
    parquet_workspace = f'{WORKSPACE_REPO_PATH}/data/processed/{parquet_filename}'

    # Strategie 1: dbutils.fs.cp cu prefix file: (sigura pentru fisiere binare)
    try:
        ws_url = f'file:/Workspace{parquet_workspace}'
        local_url = f'file:{parquet_local}'
        dbutils.fs.cp(ws_url, local_url)
        print(f'OK (dbutils.fs.cp): {parquet_local}')
    except Exception as e1:
        print(f'Strategie 1 (dbutils.fs.cp) esuata: {e1}')
        # Strategie 2: shutil.copy cu path direct /Workspace/...
        try:
            import shutil
            src = f'/Workspace{parquet_workspace}'
            shutil.copy(src, parquet_local)
            print(f'OK (shutil.copy): {parquet_local}')
        except Exception as e2:
            print(f'Strategie 2 (shutil) esuata: {e2}')
            # Strategie 3: cautare in DBFS / Volumes ca fallback
            candidates = [
                f'/dbfs/FileStore/disertatie/data/processed/{parquet_filename}',
                f'/Volumes/main/default/disertatie/data/processed/{parquet_filename}',
                f'/Volumes/workspace/default/disertatie/data/processed/{parquet_filename}',
            ]
            found = None
            for c in candidates:
                try:
                    if os.path.exists(c):
                        found = c
                        break
                except Exception:
                    continue
            if found:
                parquet_local = found
                print(f'Fisier gasit in: {parquet_local}')
            else:
                raise FileNotFoundError(
                    f'Parquet {parquet_filename} nu a fost gasit. '
                    f'Incarca-l in data/processed/ din Repo sau in /dbfs/FileStore/disertatie/data/processed/'
                )
else:
    print(f'Parquet deja in cache: {parquet_local}')

df = pd.read_parquet(parquet_local)
print(f'Shape: {df.shape}')
print(f'Range: {df.index.min()} -> {df.index.max()}')
print(f"Target: 'price actual'")
print(f'Numar features: {df.shape[1] - 1}')
print(f"\nStatistici target:")
print(df['price actual'].describe().round(2))

## 2. Regim de rulare: DEMO sau FULL

Pe Databricks recomand `MODE = "full"` direct - hardware-ul permite. Ramai pe `"demo"` doar daca vrei sa testezi rapid setup-ul (conexiuni, MLflow).

In [ ]:
MODE = "full"  # "demo" sau "full"

PARAMS = {
    "demo": {
        "N_TRAIN_LSTM": 1000, "N_VAL_LSTM": 300, "N_TEST_LSTM": 500,
        "LSTM_UNITS": 16, "LSTM_EPOCHS": 3, "LSTM_BATCH": 64,
        "OPTUNA_TRIALS": 5, "OPTUNA_SPLITS": 3, "N_TUNE": 2000,
        "SHAP_SAMPLES": 200,
        "N_CV": 5000, "CV_SPLITS": 3,
        "RF_ESTIMATORS": 30, "RF_DEPTH": 10,
        "XGB_ESTIMATORS": 100, "XGB_DEPTH": 6,
    },
    "full": {
        "N_TRAIN_LSTM": 20000, "N_VAL_LSTM": 3000, "N_TEST_LSTM": 4000,
        "LSTM_UNITS": 64, "LSTM_EPOCHS": 30, "LSTM_BATCH": 128,
        "OPTUNA_TRIALS": 50, "OPTUNA_SPLITS": 3, "N_TUNE": 20000,
        "SHAP_SAMPLES": 2000,
        "N_CV": 20000, "CV_SPLITS": 5,
        "RF_ESTIMATORS": 200, "RF_DEPTH": None,
        "XGB_ESTIMATORS": 300, "XGB_DEPTH": 6,
    },
}
P = PARAMS[MODE]
print(f"Mod activ: '{MODE}'")
print(f"  Optuna: {P['OPTUNA_TRIALS']} trials x {P['OPTUNA_SPLITS']} folduri")
print(f"  LSTM: train={P['N_TRAIN_LSTM']}, epochs={P['LSTM_EPOCHS']}")
print(f"  SHAP: {P['SHAP_SAMPLES']} samples")

## 3. Split cronologic + scaler

Cu 80 features, scalarea e CRITICA pentru modelele liniare regularizate (Ridge, Lasso) si pentru LSTM. Fara scalare, features cu scale mari ar domina absolut. `StandardScaler` se fit-eaza DOAR pe train (evitam data leakage).

In [ ]:
sp = chronological_split(
    df, target="price actual",
    test_size=P_CFG["test_size"],
    validation_size=P_CFG["validation_size"],
)
print(f"Train: {sp['X_train'].shape}, {sp['X_train'].index.min().date()} -> {sp['X_train'].index.max().date()}")
print(f"Val:   {sp['X_val'].shape}, {sp['X_val'].index.min().date()} -> {sp['X_val'].index.max().date()}")
print(f"Test:  {sp['X_test'].shape}, {sp['X_test'].index.min().date()} -> {sp['X_test'].index.max().date()}")

Xt, Xv, Xs, scaler = scale_features(sp['X_train'], sp['X_val'], sp['X_test'], method='standard')
yt, yv, ys = sp['y_train'], sp['y_val'], sp['y_test']
print(f"\nDupa scalare: train mean ~0 ({Xt.iloc[:, 0].mean():.3f}), std ~1 ({Xt.iloc[:, 0].std():.3f})")

## 4. Modele clasice + MLflow logging (Linear, Ridge, Lasso, RF, XGBoost)

### Concept nou pentru Spania - regularizarea

Cu 80 features, `LinearRegression` e expus la **multicoliniaritate** (multe features corelate - ex. suma generarii diferitelor surse aproape egala cu cererea totala).

**Ridge (L2)** adauga la loss o penalitate proportionala cu suma patratelor coeficientilor: `Loss = MSE + lambda * sum(w_i^2)`. Forteaza coeficientii sa fie mici, dar **niciodata zero**.

**Lasso (L1)** foloseste suma valorilor absolute: `Loss = MSE + lambda * sum(|w_i|)`. Forteaza coeficientii inutili **la zero exact**, facand selectie automata de features.

Fiecare model e logat ca un **MLflow run** separat.

In [ ]:
results = []

# 1. LinearRegression (baseline)
print(">>> LinearRegression")
with mlflow.start_run(run_name="LinearRegression"):
    t = time.time()
    m_lin = train_linear(Xt, yt)
    metrics = evaluate(ys, m_lin.predict(Xs))
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(m_lin, "model", input_example=Xt.head())
    results.append(ModelResult(name="LinearRegression", model=m_lin, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f} ({time.time()-t:.1f}s)")

# 2. Ridge (L2)
print(">>> Ridge (L2)")
with mlflow.start_run(run_name="Ridge"):
    t = time.time()
    mlflow.log_param("alpha", 1.0)
    m_ridge = Ridge(alpha=1.0); m_ridge.fit(Xt, yt)
    metrics = evaluate(ys, m_ridge.predict(Xs))
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(m_ridge, "model", input_example=Xt.head())
    results.append(ModelResult(name="Ridge", model=m_ridge, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f} ({time.time()-t:.1f}s)")

# 3. Lasso (L1) - feature selection automat
print(">>> Lasso (L1)")
with mlflow.start_run(run_name="Lasso"):
    t = time.time()
    mlflow.log_params({"alpha": 0.1, "max_iter": 10000})
    m_lasso = Lasso(alpha=0.1, max_iter=10000); m_lasso.fit(Xt, yt)
    metrics = evaluate(ys, m_lasso.predict(Xs))
    n_nonzero = int((m_lasso.coef_ != 0).sum())
    mlflow.log_metrics(metrics); mlflow.log_metric("features_nonzero", n_nonzero)
    mlflow.sklearn.log_model(m_lasso, "model", input_example=Xt.head())
    results.append(ModelResult(name="Lasso", model=m_lasso, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f}, features non-zero: {n_nonzero}/{len(m_lasso.coef_)} ({time.time()-t:.1f}s)")

# 4. RandomForest
print(f">>> RandomForest (n={P['RF_ESTIMATORS']}, depth={P['RF_DEPTH']})")
with mlflow.start_run(run_name="RandomForest"):
    t = time.time()
    mlflow.log_params({"n_estimators": P["RF_ESTIMATORS"], "max_depth": P["RF_DEPTH"]})
    m_rf = train_random_forest(Xt, yt, n_estimators=P["RF_ESTIMATORS"], max_depth=P["RF_DEPTH"])
    metrics = evaluate(ys, m_rf.predict(Xs))
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(m_rf, "model", input_example=Xt.head())
    results.append(ModelResult(name="RandomForest", model=m_rf, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f} ({time.time()-t:.1f}s)")

# 5. XGBoost (default)
print(f">>> XGBoost (n={P['XGB_ESTIMATORS']}, depth={P['XGB_DEPTH']})")
with mlflow.start_run(run_name="XGBoost"):
    t = time.time()
    mlflow.log_params({"n_estimators": P["XGB_ESTIMATORS"], "max_depth": P["XGB_DEPTH"]})
    m_xgb = train_xgboost(Xt, yt, n_estimators=P["XGB_ESTIMATORS"], max_depth=P["XGB_DEPTH"])
    metrics = evaluate(ys, m_xgb.predict(Xs))
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(m_xgb, "model", input_example=Xt.head())
    results.append(ModelResult(name="XGBoost", model=m_xgb, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f} ({time.time()-t:.1f}s)")

print("\n=== Sumar intermediar TEST ===")
for r in sorted(results, key=lambda r: r.r2, reverse=True):
    print(f"{r.name:<20s} RMSE={r.rmse:>8.2f} MAE={r.mae:>8.2f} R^2={r.r2:.4f} MAPE={r.mape:.2f}%")

**Cum interpretam Lasso**: daca elimina (coeficient = 0) o parte din cele 80 features pastrand performanta similara, confirma redundanta in setul de features. Numarul de features non-zero (logat ca metrica in MLflow) arata cat de "necesare" sunt features-urile.

## 5. Hyperparameter tuning cu Optuna (Bayesian Optimization)

### Ce este Optuna si de ce e mai bun decat GridSearchCV

GridSearchCV testeaza **toate** combinatiile dintr-o grila fixa (explozie combinatoriala). **Optuna** foloseste **Bayesian optimization** cu **Tree Parzen Estimator (TPE)**: dupa fiecare trial modeleaza distributia rezultatelor "bune" vs "rele" si alege urmatoarele hyperparametri inteligent. Are si **MedianPruner** care opreste trial-urile slabe devreme (economiseste 30-50% timp).

### Search space (distributii, nu valori discrete)
`n_estimators` int 100-500, `max_depth` int 3-10, `learning_rate` log 0.01-0.3, `subsample` 0.6-1.0, `colsample_bytree` 0.6-1.0.

In [ ]:
from xgboost import XGBRegressor

N_TUNE = min(P["N_TUNE"], len(Xt))
Xt_tune = Xt.iloc[-N_TUNE:]; yt_tune = yt.iloc[-N_TUNE:]

print(f">>> Optuna XGBoost pe {N_TUNE} randuri | {P['OPTUNA_TRIALS']} trials x {P['OPTUNA_SPLITS']} folduri")
with mlflow.start_run(run_name="XGBoost_tuned_Optuna"):
    mlflow.log_params({"n_tune": N_TUNE, "n_trials": P["OPTUNA_TRIALS"], "n_splits": P["OPTUNA_SPLITS"]})
    t = time.time()
    optuna_result = tune_with_optuna(
        XGBRegressor,
        param_space={
            "n_estimators": ("int", 100, 500),
            "max_depth": ("int", 3, 10),
            "learning_rate": ("float_log", 0.01, 0.3),
            "subsample": ("float", 0.6, 1.0),
            "colsample_bytree": ("float", 0.6, 1.0),
        },
        X_train=Xt_tune, y_train=yt_tune,
        n_trials=P["OPTUNA_TRIALS"], n_splits=P["OPTUNA_SPLITS"],
        direction="minimize", show_progress=False, use_pruner=True,
    )
    print(f"   Optuna terminat in {(time.time()-t)/60:.1f} min")
    print(f"   Trials completate: {optuna_result['n_trials_completed']}, pruned: {optuna_result['n_trials_pruned']}")
    print(f"   Best CV RMSE: {optuna_result['best_value']:.2f}")
    mlflow.log_params({f"best_{k}": v for k, v in optuna_result['best_params'].items()})
    mlflow.log_metric("best_cv_rmse", optuna_result['best_value'])

    # Model final cu best params, evaluat pe TEST
    m_xgb_tuned = XGBRegressor(**optuna_result['best_params'], random_state=42, n_jobs=-1)
    m_xgb_tuned.fit(Xt, yt)
    metrics_tuned = evaluate(ys, m_xgb_tuned.predict(Xs))
    mlflow.log_metrics(metrics_tuned)
    mlflow.xgboost.log_model(m_xgb_tuned, "model", input_example=Xt.head())
    results.append(ModelResult(name="XGBoost_tuned_Optuna", model=m_xgb_tuned, **metrics_tuned))
    print(f"   TEST: RMSE={metrics_tuned['rmse']:.2f}, R^2={metrics_tuned['r2']:.4f}, MAPE={metrics_tuned['mape']:.2f}%")

## 6. LSTM cu MLflow + GPU acceleration

LSTM e mai potrivit pentru Spania decat pentru USA - cu 80 features si volatilitate ridicata, capacitatea de a invata pattern-uri complexe e valoroasa. In demo, parametrii redusi dau R^2 slab (chiar negativ); in **full** pe Databricks parametrii completi dau rezultate realiste.

In [ ]:
print(">>> LSTM cu MLflow")
with mlflow.start_run(run_name="LSTM"):
    N_TRAIN_LSTM = min(P["N_TRAIN_LSTM"], len(Xt))
    N_VAL_LSTM = min(P["N_VAL_LSTM"], len(Xv))
    N_TEST_LSTM = min(P["N_TEST_LSTM"], len(Xs))
    mlflow.log_params({
        "sequence_length": 24, "units": P["LSTM_UNITS"],
        "epochs": P["LSTM_EPOCHS"], "batch_size": P["LSTM_BATCH"],
        "n_train": N_TRAIN_LSTM,
    })
    print(f"   Train: {N_TRAIN_LSTM}, Val: {N_VAL_LSTM}, Test: {N_TEST_LSTM} | units={P['LSTM_UNITS']}, epochs={P['LSTM_EPOCHS']}")
    t = time.time()
    lstm_bundle = train_lstm(
        X_train=Xt.iloc[-N_TRAIN_LSTM:].values,
        y_train=yt.iloc[-N_TRAIN_LSTM:].values,
        X_val=Xv.iloc[-N_VAL_LSTM:].values if len(Xv) >= N_VAL_LSTM else Xv.values,
        y_val=yv.iloc[-N_VAL_LSTM:].values if len(yv) >= N_VAL_LSTM else yv.values,
        sequence_length=24, units=P["LSTM_UNITS"], epochs=P["LSTM_EPOCHS"],
        batch_size=P["LSTM_BATCH"], patience=5, verbose=1,
    )
    y_pred_lstm = predict_lstm(lstm_bundle, Xs.iloc[:N_TEST_LSTM].values)
    y_test_lstm = ys.iloc[:N_TEST_LSTM].values
    mask = ~np.isnan(y_pred_lstm)
    metrics_lstm = evaluate(y_test_lstm[mask], y_pred_lstm[mask])
    mlflow.log_metrics(metrics_lstm)
    mlflow.tensorflow.log_model(lstm_bundle["model"], "model")
    results.append(ModelResult(name="LSTM", model=lstm_bundle, **metrics_lstm))
    print(f"\n   LSTM antrenat in {(time.time()-t)/60:.1f} min")
    print(f"   RMSE={metrics_lstm['rmse']:.2f}, R^2={metrics_lstm['r2']:.4f}, MAPE={metrics_lstm['mape']:.2f}%")

## 7. Tabel comparativ final + grafic predictii

In [ ]:
df_comp = pd.DataFrame([r.to_dict() for r in results]).sort_values("r2", ascending=False).reset_index(drop=True)
print("=== Comparatie modele Spania (sortat dupa R^2) ===")
print(df_comp.to_string(index=False))
print(f"\nCastigator: {df_comp.iloc[0]['model']}")
df_comp

### Plot: predictii vs real - ultima saptamana din test

In [ ]:
last_week_idx = ys.index[-168:]
y_true_lw = ys.iloc[-168:].values

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(last_week_idx, y_true_lw, label="Real", color="black", lw=2.0, alpha=0.9)
colors = {"LinearRegression": PALETA["primary"], "Ridge": "#5B89C7", "Lasso": "#7BA4DC",
          "RandomForest": PALETA["secondary"], "XGBoost": PALETA["tertiary"],
          "XGBoost_tuned_Optuna": "#E11D48", "LSTM": PALETA["neutral"]}
for r in results:
    try:
        if r.name in ("LinearRegression", "Ridge", "Lasso", "RandomForest", "XGBoost", "XGBoost_tuned_Optuna"):
            y_pred_lw = r.model.predict(Xs.iloc[-168:])
            ax.plot(last_week_idx, y_pred_lw, label=r.name, color=colors.get(r.name, "gray"), lw=1.0, alpha=0.7)
    except Exception:
        pass
ax.set_title("Spania - Predictii vs Real - ultima saptamana din test (Databricks)")
ax.set_ylabel("Pret (EUR/MWh)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
FIG_PRED = '/tmp/fig_6_pred_vs_real_spania.png'
plt.savefig(FIG_PRED, dpi=130, bbox_inches='tight')
plt.show()

## 8. Feature importance (XGBoost tunat)

In [ ]:
fi = get_feature_importance(m_xgb_tuned, Xt.columns)
print("Top 20 features (XGBoost optimizat - Spania):")
print(fi.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 7))
top20 = fi.head(20)
ax.barh(top20['feature'][::-1], top20['importance'][::-1], color=PALETA["primary"], edgecolor="black")
ax.set_xlabel("Importance")
ax.set_title("Top 20 features - XGBoost tunat (Spania)")
plt.tight_layout()
FIG_FI = '/tmp/fig_6_feature_importance_spania.png'
plt.savefig(FIG_FI, dpi=130, bbox_inches='tight')
plt.show()

## 9. SHAP values pentru explicabilitate

### Ce sunt valorile Shapley

In **teoria jocurilor cooperative** (Lloyd Shapley, Nobel 2012), valorile Shapley impart echitabil "castigul" intre membrii unei coalitii. Pentru ML, fiecare feature e un "membru" care contribuie la predictie.

**Avantaj fata de `feature_importances_`**: acesta da o singura valoare globala per feature. SHAP da o valoare per (predictie, feature) - vedem cat a contribuit fiecare feature la fiecare predictie individuala, si DIRECTIA efectului (creste sau scade pretul).

Pe Databricks, calculul SHAP ruleaza pe mai multe sample-uri (full: 2000) si plot-ul e logat ca artifact MLflow.

In [ ]:
print(f"Calculez SHAP values pe {P['SHAP_SAMPLES']} sample-uri din test...")
t = time.time()
shap_result = compute_shap_values(m_xgb_tuned, Xs, max_samples=P['SHAP_SAMPLES'])
print(f"Calcul terminat in {time.time()-t:.1f}s")
print(f"Shape SHAP values: {shap_result['shap_values'].shape}")
print(f"Expected value (predictia medie): {shap_result['expected_value']:.2f} EUR/MWh")
print(f"Tipul explainer-ului: {shap_result['model_type']}")

In [ ]:
shap_values = shap_result['shap_values']
X_sample = shap_result['X_sample']

shap_importance = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
    'mean_shap': shap_values.mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)

print("Top 15 features dupa SHAP (importanta + directia medie):")
print(shap_importance.head(15).to_string(index=False))

top15 = shap_importance.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 7))
colors_shap = ['#E11D48' if x > 0 else '#0891B2' for x in top15['mean_shap']]
ax.barh(top15['feature'], top15['mean_abs_shap'], color=colors_shap, edgecolor="black")
ax.set_xlabel("|SHAP value| mediu - importanta absoluta")
ax.set_title("Top 15 features - SHAP analysis (Spania)\nRosu = creste pretul in medie | Albastru = scade pretul")
plt.tight_layout()
FIG_SHAP = '/tmp/fig_6_shap_summary_spania.png'
plt.savefig(FIG_SHAP, dpi=130, bbox_inches='tight')
plt.show()

## 10. Salvare rezultate in output dir + export inapoi in GitHub

Modelul castigator, tabelul comparativ, log-ul de training si figurile pentru Capitolul 6 se salveaza in primul director scriabil (preferinta: workspace files in repo, ca sa se sincronizeze prin git).

In [ ]:
def find_writable_output_dir(project_root: Path) -> Path:
    candidates = [
        project_root,
        Path("/dbfs/FileStore/disertatie"),
        Path("/Volumes/main/default/disertatie"),
        Path("/Volumes/workspace/default/disertatie"),
    ]
    for c in candidates:
        try:
            (c / "models").mkdir(parents=True, exist_ok=True)
            (c / "reports" / "figures").mkdir(parents=True, exist_ok=True)
            test = c / "reports" / ".test_write"
            test.write_text("ok"); test.unlink()
            return c
        except Exception:
            continue
    raise RuntimeError("Niciun director nu este scriabil pentru output.")

OUTPUT_DIR = find_writable_output_dir(PROJECT_ROOT)
print(f"Output dir: {OUTPUT_DIR}")

# Castigator (dintre modelele cu API standard - excludem LSTM care e bundle)
classic_results = [r for r in results if r.name != "LSTM"]
classic_results.sort(key=lambda r: r.r2, reverse=True)
winner = classic_results[0]
print(f"Castigator (clasic): {winner.name} cu R^2 = {winner.r2:.4f}")

if "XGB" in type(winner.model).__name__:
    save_path = OUTPUT_DIR / "models" / "spania_winner_xgboost_databricks.json"
    save_model(winner.model, save_path, kind="xgboost")
else:
    save_path = OUTPUT_DIR / "models" / "spania_winner_databricks.pkl"
    save_model(winner.model, save_path, kind="sklearn")
print(f"Salvat: {save_path}")

# Tabel comparativ
df_final = pd.DataFrame([r.to_dict() for r in results]).sort_values("r2", ascending=False)
df_final.insert(0, "dataset", "pret_spania")
df_final.insert(1, "platform", "databricks")
csv_path = OUTPUT_DIR / "reports" / "ml_comparison_spania_databricks.csv"
df_final.to_csv(csv_path, index=False)
print(f"Tabel salvat: {csv_path}")

# Copiez figurile in reports/figures
import shutil as _sh
for src_fig, dst_name in [
    (FIG_PRED, 'fig_6_1_pred_vs_real_spania.png'),
    (FIG_FI,   'fig_6_2_feature_importance_spania.png'),
    (FIG_SHAP, 'fig_6_3_shap_summary_spania.png'),
]:
    try:
        _sh.copy(src_fig, OUTPUT_DIR / "reports" / "figures" / dst_name)
        print(f"Figura salvata: reports/figures/{dst_name}")
    except Exception as e:
        print(f"FAIL figura {dst_name}: {e}")
df_final

## 10.5. Salvare training log detaliat

Salvam toate detaliile importante intr-un fisier markdown care ajunge pe git. AI-ul il va citi in sesiunea urmatoare ca sa scrie Capitolul 6 din lucrare.

In [ ]:
import datetime

log = []
log.append('# Training Log - Sesiunea 2 Spania pe Databricks')
log.append('')
log.append('Generat: ' + datetime.datetime.now().isoformat())
log.append('Mod rulare: **' + str(MODE) + '**')
log.append('Cluster: UTM Databricks (16.4 LTS)')
log.append('Output dir: ' + str(OUTPUT_DIR))

log.append('')
log.append('## 1. Tabel comparativ modele')
log.append('```')
log.append(df_final.to_string(index=False))
log.append('```')
log.append('')
log.append('**Castigator**: ' + str(df_final.iloc[0]['model']))
log.append('**RMSE castigator**: ' + f"{df_final.iloc[0]['rmse']:.2f}")
log.append('**R^2 castigator**: ' + f"{df_final.iloc[0]['r2']:.4f}")

log.append('')
log.append('## 2. Optuna best params')
try:
    log.append('```python')
    log.append(str(optuna_result['best_params']))
    log.append('```')
    log.append('Best CV RMSE: ' + f"{optuna_result['best_value']:.2f}")
    log.append('Trials completate: ' + str(optuna_result['n_trials_completed']) + ', pruned: ' + str(optuna_result['n_trials_pruned']))
except NameError:
    log.append('Optuna nu a rulat')

log.append('')
log.append('## 3. Feature importance top 30 (XGBoost tunat)')
try:
    log.append('```')
    log.append(get_feature_importance(m_xgb_tuned, Xt.columns).head(30).to_string(index=False))
    log.append('```')
except Exception as e:
    log.append('Eroare feature importance: ' + str(e))

log.append('')
log.append('## 4. SHAP top 20 (mean_abs_shap + directie)')
try:
    log.append('```')
    log.append(shap_importance.head(20).to_string(index=False))
    log.append('```')
    log.append('Expected value (predictia medie): ' + f"{shap_result['expected_value']:.2f} EUR/MWh")
except Exception as e:
    log.append('Eroare SHAP: ' + str(e))

log.append('')
log.append('## 5. Lasso - features non-zero')
try:
    log.append('Features non-zero: ' + str(int((m_lasso.coef_ != 0).sum())) + '/' + str(len(m_lasso.coef_)))
except Exception as e:
    log.append('Eroare Lasso: ' + str(e))

log.append('')
log.append('## 6. LSTM training history')
try:
    if hasattr(lstm_bundle.get('model'), 'history'):
        hist = lstm_bundle['model'].history.history
        log.append('Epochs antrenate: ' + str(len(hist.get('loss', []))))
        log.append('```')
        log.append('Epoch    Loss        Val Loss')
        for i in range(len(hist.get('loss', []))):
            ml = hist['loss'][i]
            vl = hist.get('val_loss', [None]*len(hist['loss']))[i] if 'val_loss' in hist else None
            vl_str = f'{vl:.4f}' if isinstance(vl, float) else 'N/A'
            log.append(f'{i+1:<9}{ml:<12.4f}{vl_str:<12}')
        log.append('```')
    else:
        log.append('LSTM history nu este disponibil')
except Exception as e:
    log.append('Eroare LSTM history: ' + str(e))

log.append('')
log.append('## 7. Setari utilizate')
log.append('```python')
for k, v in P.items():
    log.append(str(k) + ': ' + str(v))
log.append('```')

log.append('')
log.append('## 8. Date despre dataset')
log.append('- Total randuri: ' + str(len(df)))
log.append('- Numar features: ' + str(df.shape[1] - 1))
log.append('- Train: ' + str(len(sp['X_train'])) + ', Val: ' + str(len(sp['X_val'])) + ', Test: ' + str(len(sp['X_test'])))

log_path = OUTPUT_DIR / 'reports' / 'training_log_spania_databricks.md'
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text(chr(10).join(log), encoding='utf-8')
print('Training log salvat: ' + str(log_path) + ' (' + str(len(log)) + ' linii)')
print('')
print(chr(10).join(log[:30]))

## 10.6. Copiere rezultate inapoi in Workspace Files (pentru git)

Aducem output-urile din `/tmp/` inapoi in repo-ul din Workspace, ca sa apara in dialogul Git si sa le poti commita / pulla local.

In [ ]:
import shutil

LOCAL_OUTPUT = '/tmp/disertatie_ai_platform'
WORKSPACE_BASE = '/Workspace' + WORKSPACE_REPO_PATH

files_to_copy = [
    'models/spania_winner_xgboost_databricks.json',
    'models/spania_winner_databricks.pkl',
    'reports/ml_comparison_spania_databricks.csv',
    'reports/training_log_spania_databricks.md',
    'reports/figures/fig_6_1_pred_vs_real_spania.png',
    'reports/figures/fig_6_2_feature_importance_spania.png',
    'reports/figures/fig_6_3_shap_summary_spania.png',
]

for f_rel in files_to_copy:
    src = LOCAL_OUTPUT + '/' + f_rel
    dst = WORKSPACE_BASE + '/' + f_rel
    if not os.path.exists(src):
        print('SKIP (nu exista in /tmp): ' + f_rel)
        continue
    try:
        dbutils.fs.cp('file:' + src, 'file:' + dst)
        print('OK: ' + f_rel + ' -> workspace')
    except Exception as e:
        try:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
            print('OK shutil: ' + f_rel + ' -> workspace')
        except Exception as e2:
            print('FAIL: ' + f_rel + ': ' + str(e2))

print('')
print('Done! Verifica dialogul Git - ar trebui sa apara fisierele noi.')

## 11. Concluzii Sesiunea 2 (Databricks, full)

**Ce s-a obtinut:**
- Pipeline complet pentru pretul energiei Spania pe 80 features, cu 7 modele comparate si logate in MLflow.
- **Optuna** (Bayesian) ca metoda de tuning - alternativa moderna la GridSearchCV.
- **SHAP values** - explicabilitate granulara, cu directia efectului fiecarui feature asupra pretului.
- Model castigator + tabel comparativ + training log + 3 figuri pentru Capitolul 6, salvate in repo.

**Pas urmator:** dupa pull local, AI-ul scrie **Capitolul 6** din `Disertatie.docx` din `training_log_spania_databricks.md` si figurile generate. Apoi Sesiunea 3 (Solar India).

**Vizualizare experimente:** tab-ul "Experiments" din Databricks UI - compara toate run-urile side-by-side (parametri + metrici).